In [1]:
!pip install pandas scikit-learn openpyxl

In [2]:
import pandas as pd

df = pd.read_excel("singlish_comments_2500_labeled.xlsx", sheet_name="Dataset")
df = df.dropna(subset=["Label"])
df["Label"] = df["Label"].astype(int)

print(f"Total comments: {len(df)}")
print(f"Hate (1): {sum(df['Label'] == 1)}")
print(f"Not Hate (0): {sum(df['Label'] == 0)}")

Total comments: 2500
Hate (1): 1305
Not Hate (0): 1195


In [3]:
from sklearn.model_selection import train_test_split

X = df["Comment"].astype(str)
y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)}")
print(f"Test set: {len(X_test)}")

Training set: 2000
Test set: 500


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"TF-IDF features: {X_train_tfidf.shape[1]}")

TF-IDF features: 1796


In [5]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
    "Naive Bayes": MultinomialNB(),
    "SVM": LinearSVC(max_iter=2000),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)

    print(f"\n{name}:")
    print(f"  Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"  Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
    print(f"  Recall:    {recall_score(y_test, y_pred, average='weighted'):.4f}")
    print(f"  F1-Score:  {f1_score(y_test, y_pred, average='weighted'):.4f}")


Naive Bayes:
  Accuracy:  0.8120
  Precision: 0.8128
  Recall:    0.8120
  F1-Score:  0.8115

SVM:
  Accuracy:  0.8040
  Precision: 0.8041
  Recall:    0.8040
  F1-Score:  0.8040

Logistic Regression:
  Accuracy:  0.8120
  Precision: 0.8137
  Recall:    0.8120
  F1-Score:  0.8113

Random Forest:
  Accuracy:  0.7900
  Precision: 0.7907
  Recall:    0.7900
  F1-Score:  0.7894


In [6]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tfidf_cv = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=5000)
X_tfidf_all = tfidf_cv.fit_transform(X)

for name, model in models.items():
    scores = cross_val_score(model, X_tfidf_all, y, cv=cv, scoring='f1_weighted')
    print(f"{name}: F1 = {scores.mean():.4f} ± {scores.std():.4f}")

Naive Bayes: F1 = 0.8462 ± 0.0085
SVM: F1 = 0.8324 ± 0.0081
Logistic Regression: F1 = 0.8403 ± 0.0152
Random Forest: F1 = 0.8134 ± 0.0095


In [7]:
# Save baseline results
baseline_results = pd.DataFrame({
    "Model": ["Naive Bayes", "Logistic Regression", "SVM", "Random Forest"],
    "Accuracy": [0.8120, 0.8120, 0.8040, 0.7900],
    "F1_Score": [0.8115, 0.8113, 0.8040, 0.7894],
    "CV_F1_Mean": [0.8462, 0.8403, 0.8324, 0.8134],
    "CV_F1_Std": [0.0085, 0.0152, 0.0081, 0.0095]
})

baseline_results.to_csv("baseline_results.csv", index=False)
print("Baseline results saved!")
print(baseline_results)

Baseline results saved!
                 Model  Accuracy  F1_Score  CV_F1_Mean  CV_F1_Std
0          Naive Bayes     0.812    0.8115      0.8462     0.0085
1  Logistic Regression     0.812    0.8113      0.8403     0.0152
2                  SVM     0.804    0.8040      0.8324     0.0081
3        Random Forest     0.790    0.7894      0.8134     0.0095


In [1]:
!pip install transformers torch datasets

In [2]:
import torch
from torch.utils.data import Dataset

class SinglishDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = self.labels.iloc[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": torch.tensor(label, dtype=torch.long)
        }

print("Dataset class created!")

Dataset class created!


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "NLPC-UOM/SinBERT-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print("SinBERT-small loaded!")
print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters")

config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/839k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/639k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  266MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: NLPC-UOM/SinBERT-small
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


SinBERT-small loaded!
Model has 66,556,418 parameters


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "NLPC-UOM/SinBERT-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print("SinBERT-small loaded!")

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: NLPC-UOM/SinBERT-small
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


SinBERT-small loaded!


In [6]:
import torch
from torch.utils.data import Dataset

class SinglishDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = self.labels.iloc[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": torch.tensor(label, dtype=torch.long)
        }

print("Dataset class created!")

Dataset class created!


In [8]:
!pip install pandas scikit-learn openpyxl transformers torch datasets

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_excel("singlish_comments_2500_labeled.xlsx", sheet_name="Dataset")
df = df.dropna(subset=["Label"])
df["Label"] = df["Label"].astype(int)

X = df["Comment"].astype(str)
y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Total: {len(df)}")
print(f"Train: {len(X_train)}")
print(f"Test: {len(X_test)}")

Total: 2500
Train: 2000
Test: 500


In [10]:
import torch
from torch.utils.data import Dataset

class SinglishDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = self.labels.iloc[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": torch.tensor(label, dtype=torch.long)
        }

print("Dataset class created!")

Dataset class created!


In [11]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "NLPC-UOM/SinBERT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

print("SinBERT-small loaded!")

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: NLPC-UOM/SinBERT-small
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


SinBERT-small loaded!


In [12]:
train_dataset = SinglishDataset(X_train, y_train, tokenizer)
test_dataset = SinglishDataset(X_test, y_test, tokenizer)

print(f"Train dataset: {len(train_dataset)}")
print(f"Test dataset: {len(test_dataset)}")

Train dataset: 2000
Test dataset: 500


In [13]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

training_args = TrainingArguments(
    output_dir="./sinbert_results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.696259,0.697908,0.584000,0.485328
2,0.589713,0.604934,0.656000,0.621132
3,0.529150,0.539730,0.738000,0.735680


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=375, training_loss=0.6032713521321614, metrics={'train_runtime': 88.0248, 'train_samples_per_second': 68.163, 'train_steps_per_second': 4.26, 'total_flos': 198701097984000.0, 'train_loss': 0.6032713521321614, 'epoch': 3.0})

In [14]:
results = trainer.evaluate()
print(f"SinBERT F1-Score: {results['eval_f1']:.4f}")
print(f"SinBERT Accuracy: {results['eval_accuracy']:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.529150,0.539730,3,0.738000,0.735680


SinBERT F1-Score: 0.7357
SinBERT Accuracy: 0.7380
